# పాఠం 13 - ఏజెంట్ మెమరీ


## సెటప్

ఈ నోట్‌బుక్ **Microsoft Agent Framework** (MAF) ఉపయోగించి **స్థిరమైన జ్ఞాపకం** కలిగిన ప్రయాణ బుకింగ్ ఏజెంట్‌ను ఎలా నిర్మించాలో చూపిస్తుంది.

ఏజెంట్ జ్ఞాపకం యొక్క వివిధ రకాలైన — వర్కింగ్, షార్ట్-టర్మ్, మరియు లాంగ్-టర్మ్ — ఏవీ ఏజెంట్ సంభాషణల మధ్య సమాచారం నిలుపుకోవడంలో మరియు ఉపయోగించడంలో ఎలా ప్రభావం చూపిస్తాయో మీరు తెలుసుకుంటారు.

**అవసరాలు:**
- ఒక Microsoft Foundry ప్రాజెక్ట్, అందులో ఒక చాట్ మోడల్ (ఉదాహరణకు `gpt-4.1-mini`) డిప్లాయ్ చేసి ఉండాలి.
- Azure CLI లో లాగిన్ అయిన ఉండాలి — టెర్మినల్‌లో `az login` ఆదేశం నడిపించండి.
- `AZURE_AI_PROJECT_ENDPOINT` — మీ Microsoft Foundry ప్రాజెక్ట్ ఎండ్‌పాయింట్.
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — మీరు డిప్లాయ్ చేసిన మోడల్ పేరు.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import json
import dotenv
from typing import Annotated
from datetime import datetime

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## ఏజెంట్ మెమరీ రకాలు

AI ఏజెంట్లు వేర్వేరు రకాల మెమరీలను ఉపయోగించవచ్చు, ప్రతి ఒకటి ప్రత్యేక ప్రయోజనం కోసం పనిచేస్తుంది:

### వర్కింగ్ మెమరీ
సంభాషణ తంతువు స్వయంగా — ఒక సెషన్ లో మార్చుకున్న సందేశాలు. ఏజెంట్ coherenceని మెయింటైన్ చేసుకోవడానికి అదే తంతువు లో పూర్వ సందేశాలను స్మరించుకోవచ్చు. MAF లో ఇది **`agent.create_session()`** ద్వారా సృష్టించబడుతుంది, ఇది `AgentSession`ని రిటర్న్ చేస్తుంది.

### షార్ట్-టర్మ్ మెమరీ
ఒక టాస్క్ లేదా సెషన్ దాకా నిలిచే సమాచారం కాని శాశ్వతంగా నిల్వ కాదు. ఉదాహరణకి, ఏజెంట్ బహుళ-టర్న్ ప్లానింగ్ సంభాషణలో ఫాక్ట్స్ సేకరించి వాటిని ఉపయోగించి తుది యాత్రా ప్రణాళికను తయారు చేస్తుంది.

### లాంగ్-టర్మ్ మెమరీ
సెషన్ల మధ్య కలిగే అభిరుచులు మరియు వాస్తవాలు. తిరిగి వచ్చిన యూజర్ ఆహార పరిమితులు లేదా ప్రయాణ శైలి పునరావృతం చేయాల్సిన అవసరము ఉండకూడదు. లాంగ్-టర్మ్ మెమరీ సాధారణంగా బాహ్య నిల్వ ద్వారా రక్షించబడుతుంది — డేటాబేస్, ఫైల్, లేదా వెక్టర్ సూచిక — మరియు టూల్స్ ద్వారా ఏజెంట్‌కి అందించబడుతుంది.


## సెషన్లతో పని చేసే మెమొరీ

మెమోరీ యొక్క సర్వసాధారణ రకం సంభాషణ సెషన్. మీరు అదే సెషన్‌ను (`agent.create_session()` ఉపయోగించి సృష్టించబడింది) వరుసగా `agent.run()` కాల్స్‌కు పంపినప్పుడు, ఏజెంట్ ఆ సంభాషణను పూర్తిగా చూసి, ప్రారంభ వివరాలను గుర్తు చేయగలదు.

మనం ఒక ప్రయాణ ఏజెంట్ ను సృష్టించి పని చేసే మెమొరీని ప్రదర్శిద్దాం.


In [ ]:
agent = client.as_agent(
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

ఏజెంట్ బడ్జెట్‌ను సరైనంగా గుర్తుచేశాడు ఎందుకంటే రెండు సందేశాలు ఒకే సెషన్‌ను పంచుకుంటున్నాయి. ఇది **వర్తించే స్మృతి** — ఇది సెషన్ జీవితకాలం వరకు మాత్రమే ఉనికిలో ఉంటుంది.

### కొత్త థ్రెడ్‌తో ఏమి జరుగుతుంది?

మనం ఒక **కొత్త** సెషన్ సృష్టిస్తే, ఏజెంట్ కు గత సంభాషణ యొక్క స్మృతి ఉండదు:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## దీర్ఘకాలిక జ్ఞాపక నమూనా

వినియోగదారు ప్రాధాన్యతలను **సెషన్ల మధ్య** గుర్తుంచుకోవడానికి, సంభాషణ థ్రెడ్ వెలుపల ఉండే ఒక స్థిరమైన నిల్వ అవసరం. ఏజెంట్ ఈ నిల్వను **ఉపకరణాలు** ద్వారా యాక్సెస్ చేస్తుంది — సేవ్ చేయడం మరియు సమాచారం పొందడం కోసం పిలిచే ఫంక్షన్లు.

క్రింద మనం ఒక సాధారణ ఇన్-మెమరీ ప్రాధాన్యత నిల్వను అమలు చేస్తాము (ఉత్పత్తిలో దీన్ని డేటాబేస్ లేదా వెక్టర్ ఇండెక్స్‌తో బ్యాక్ చేయవచ్చు) మరియు దానిని ఏజెంట్ ఉపయోగించగల ఉపకరణాలుగా ప్రదర్శిస్తాము.

### నిర్మాణం
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### సందర్భం 1 — మొదటి సారి వినియోగదారు వార్షికోత్సవ యాత్రను బుక్ చేసుకొంటున్నారు

సారా మొదటి సారి సందర్శిస్తుంది. ఏజెంట్ ఆమె అభిరుచులను టూల్స్ ద్వారా నిల్వ చేయాలి మరియు వాటిని హోటళ్లు సూచించడానికి ఉపయోగించాలి.


In [ ]:
travel_agent = client.as_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### పరిస్థితి 2 — సారా వారాల తరువాత తిరిగి వస్తుంది

సారా ఒక **కొత్త థ్రెడ్** ప్రారంభిస్తుంది (కొత్త సెషన్‌ను అనుకరించడం). వర్కింగ్ మెమరీ ఖాళీగా ఉంటుంది, కానీ దీర్ఘకాలిక ప్రాధాన్యత స్టోర్‌లో ఆమె సమాచారం ఇంకా ఉంటుంది. ఏజెంట్ దాన్ని తిరిగి అందుకొని వ్యక్తిగత సిఫారసులను ఇవ్వాలి.


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## సారాంశం

ఈ పాఠంలో మీరు మూడు రకాల ఏజెంట్ మెమరీలు మరియు Microsoft Agent Framework తో వాటిని ఎలా అమలు చేయాలో నేర్చుకున్నారు:

| మెమరీ రకం | MAF యంత్రాంగం | జీవితకాలం |
|---|---|---|
| **పనిచేసే మెమరీ** | `agent.create_session()` | ఒక సంభాషణ |
| **తాత్కాలిక మెమరీ** | ఒక థ్రెడ్ లో సేకరించిన కంటెక్స్ట్ | ఒక టాస్క్ / సెషన్ |
| **దీర్ఘకాలిక మెమరీ** | `@tool` ఫంక్షన్ల ద్వారా యాక్సెస్ అయ్యే బయటి భాండారం | సెషన్ల మధ్య |

### ముఖ్యమైన విషయాలు
1. **`agent.create_session()`** పనిచేసే మెమరీ అందిస్తుంది — ఏజెంట్ సెషన్ లో పూర్తి సంభాషణ చరిత్రను చూస్తుంది.
2. **కొత్త సెషన్లు కంటెక్స్ట్ కోల్పోతాయి** — దీర్ఘకాలిక మెమరీ లేకపోవడం వల్ల ఏజెంట్ గత సంభాషణలను గుర్తు చేసుకోలేడు.
3. **`@tool` ఫంక్షన్లు** వక్తవ్యాన్ని లోపలుండి బయటకు కనెక్ట్ చేయడం చేస్తాయి — అర్థం ఏజెంట్ సమాచారం నిల్వ చేసి తిరిగి పొందేందుకు వీలుగా చేస్తాయి.
4. **వ్యక్తిగతీకరణ సమయం తో మెరుగవుతుంది** — ఎక్కువ అభిరుచులు స్టోర్ చేస్తే, ఏజెంట్ సిఫారసులు మరింత మంచిది అవుతాయి.

### వాస్తవ ప్రపంచంలో వినియోగాలు
- **గ్రాహక సేవలు**: కస్టమర్ చరిత్ర మరియు అభిరుచులను గుర్తుంచడం
- **వ్యక్తిగత సహాయకులు**: రోజులు లేదా వారాలుగా కంటెక్స్ట్ తీసుకోవడం
- **ఆరోగ్య సంరక్షణ**: రోగి సమాచారాన్ని మరియు అభిరుచులను ట్రాక్ చేయడం
- **ఇ-కామర్స్**: చరిత్ర ఆధారంగా వ్యక్తిగత షాపింగ్

### తదుపరి దశలు
- ఇన్-మెమరీ డిక్ట్ ను డేటాబేస్ లేదా వెక్టర్ స్టోర్ (ఉదా: Azure AI Search) తో మారుస్తూ ఉండండి
- సమయానుకూలమైన సమాచారానికి మెమరీ గడువు చేర్పు
- పంచుకోబడిన మెమరీతో బహుళ ఏజెంట్ సిస్టమ్స్ నిర్మించండి
- Cognee నోట్బుక్ లోని జ్ఞాన-గ్రాఫ్ ఆధారిత మెమరీని అన్వేషించండి


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**అస్వీకరణ**:
ఈ పత్రం AI అనువాద సేవ [Co-op Translator](https://github.com/Azure/co-op-translator) ఉపయోగించి అనువదించబడింది. మేము ఖచ్చితత్వానికి ప్రయత్నిస్తున్నప్పటికీ, ఆటోమేటెడ్ అనువాదాలు తప్పులు లేదా అసమగ్రతలను కలిగి ఉండవచ్చు. దాని స్వదేశ భాషలో ఉన్న అసలు పత్రాన్ని అధికారం కలిగిన మూలంగా పరిగణించాలి. కీలకమైన సమాచారం కోసం, ప్రొఫెషనల్ మానవ అనువాదాన్ని సిఫారసు చేస్తాము. ఈ అనువాదం ఉపయోగం వల్ల కలిగే ఏవైనా అపార్థాలు లేదా తప్పుదారులు కోసం మేము బాధ్యత వహించము.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
